This project is about analyzing current AI research trends. I inspected the first query (api url), it returned data for future-dated years, because it was sorted only by newest publication date. So now the query has been refined so that the project focuses on publications from 2020 up to today's date.

The workflow of my project is:
API → raw JSON → selected fields → DataFrame → validation → cleaning

In [9]:
#API fetch function
#The end date is generated with `datetime` library so that future publication dates are not intentionally requested.

import requests
from datetime import date


def fetch_works(api_url):
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from API: {e}")
        return None

current_date = date.today().isoformat()

api_url = (
    "https://api.openalex.org/works"
        "?search=artificial%20intelligence"
        f"&filter=from_publication_date:2020-01-01,to_publication_date:{current_date}"
        "&per_page=100"
)
works_data = fetch_works(api_url)




In [10]:
#inspecting the API response, OpenAlex returns a dictionary. The actual research works are stored inside the `results` list.

if works_data is not None:
    print(type(works_data))
    print(works_data.keys())
    print(type(works_data["results"]))
    print(len(works_data["results"]))
    print(works_data["results"][0]["title"])
    print(works_data["results"][0]["publication_year"])
    print(works_data["results"][0]["cited_by_count"])
    print(works_data["results"][0]["open_access"])

<class 'dict'>
dict_keys(['meta', 'results', 'group_by'])
<class 'list'>
100
Artificial Intelligence in Education: A Review
2020
3958
{'is_oa': True, 'oa_status': 'gold', 'oa_url': 'https://doi.org/10.1109/access.2020.2988510', 'any_repository_has_fulltext': True}


In [11]:
# saving the fetched raw data to a JSON file
import json

if works_data is not None:
    with open("data/works_data.json", "w", encoding="utf-8") as file:
        json.dump(works_data, file, indent=4)

The OpenAlex response contains many nested fields.  
For the first version of the project, only four fields are selected:

- title
- publication year
- citation count
- open-access status

In [12]:
#extracting the relevant information from the works data

results = works_data["results"]

extracted_data = []

for work in results:
    extracted_data.append({
        "title": work["title"],
        "publication_year": work["publication_year"],
        "cited_by_count": work["cited_by_count"],
        "is_open_access": work["open_access"]["is_oa"]
        
    })

print(len(extracted_data))
print(extracted_data[0])

100
{'title': 'Artificial Intelligence in Education: A Review', 'publication_year': 2020, 'cited_by_count': 3958, 'is_open_access': True}


In [13]:
#converting to a dataframe

import pandas as pd

df = pd.DataFrame(extracted_data)

print(df.head())
print(df.info())

                                               title  publication_year  \
0     Artificial Intelligence in Education: A Review              2020   
1  The role of artificial intelligence in achievi...              2020   
2  Revolutionizing healthcare: the role of artifi...              2023   
3  BNAI, NO-TOKEN, and MIND-UNITY: Pillars of a S...              2022   
4  Artificial Intelligence and Management: The Au...              2021   

   cited_by_count  is_open_access  
0            3958            True  
1            3221            True  
2            3130            True  
3            4331            True  
4            2491            True  
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   title             100 non-null    str  
 1   publication_year  100 non-null    int64
 2   cited_by_count    100 non-null    int64
 3   is_open_access   

Validate the data

Before cleaning, inspect the dataset for:

- publication-year range
- missing values
- complete duplicate rows
- duplicate titles
- number of rows and columns

This helps decide what actually needs cleaning instead of applying cleaning steps blindly.

In [14]:
print(f"Minimum publication year: {df['publication_year'].min()}")
print(f"Maximum publication year: {df['publication_year'].max()}")

print("\nMissing values per column:")
print(df.isnull().sum())

print(f"\nNumber of duplicated rows: {df.duplicated().sum()}")
print(f"Number of duplicated titles: {df['title'].duplicated().sum()}")
print(f"Shape of the DataFrame: {df.shape}")

Minimum publication year: 2020
Maximum publication year: 2024

Missing values per column:
title               0
publication_year    0
cited_by_count      0
is_open_access      0
dtype: int64

Number of duplicated rows: 0
Number of duplicated titles: 0
Shape of the DataFrame: (100, 4)


In [15]:
# Displaying all duplicate rows in the dataframe
duplicate_rows = df[df.duplicated(keep=False)]

display(duplicate_rows.head(10))

,title,publication_year,cited_by_count,is_open_access


Save validated data

The selected OpenAlex data did not contain missing values or duplicate rows.
After validation, the dataset is saved as CSV for further analysis. (I saved the original API response as JSON. After selecting and validating the fields needed for analysis, I saved the prepared tabular dataset as CSV.)

In [18]:
df.to_csv(
    "data/cleaned_works.csv",
    index=False
)

1. How many publications are there per year?
2. What is the average citation count?
3. How many works are open access vs not open access?

In [19]:
publications_per_year = (
    df["publication_year"]
    .value_counts()
    .sort_index()
)

print(publications_per_year)

publication_year
2020    38
2021    24
2022    16
2023    20
2024     2
Name: count, dtype: int64


In [20]:
average_citations = df["cited_by_count"].mean()

print(f"Average citations: {average_citations:.2f}")

Average citations: 1314.86


In [21]:
open_access_counts = df["is_open_access"].value_counts()

print(open_access_counts)

is_open_access
True     86
False    14
Name: count, dtype: int64
